## Notebook del Processing Job
El notebook debe cubrir:

- Setup: sesión de SageMaker, IAM role, bucket y prefix.
- Carga del dataset a S3: sube tus datos crudos al bucket de SageMaker.
- Ejecución del Processing Job.
- Inspección del output: lee las primeras filas del CSV transformado desde S3 para verificar que el job fue exitoso.


In [1]:
## Setup: SageMaker Session, IAM Role y Bucket S3
from time import gmtime, strftime
import sagemaker

sagemaker_session = sagemaker.Session()
role = sagemaker.get_execution_role()
bucket = sagemaker_session.default_bucket()
default_bucket_prefix = sagemaker_session.default_bucket_prefix
timestamp_prefix = strftime("%Y-%m-%d-%H-%M-%S", gmtime())

# Configurar rutas S3 para el Processing Job
prefix = "sagemaker/processing-data"

# If a default bucket prefix is specified, append it to the s3 path
if default_bucket_prefix:
    prefix = f"{default_bucket_prefix}/{prefix}"

# S3 paths para el Processing Job
input_prefix = prefix + "/input/raw/"
input_preprocessed_prefix = prefix + "/input/preprocessed/"

output_prefix = prefix + "/output"

# Rutas dentro del container del Processing Job
input_container_path = "/opt/ml/processing/input"
output_container_path = "/opt/ml/processing/output"

print(f"Bucket: {bucket}")
print(f"Input S3 path: s3://{bucket}/{input_prefix}")
print(f"Output S3 path: s3://{bucket}/{output_prefix}")




sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
Bucket: sagemaker-us-east-1-956463123241
Input S3 path: s3://sagemaker-us-east-1-956463123241/sagemaker/processing-data/input/raw/
Output S3 path: s3://sagemaker-us-east-1-956463123241/sagemaker/processing-data/output


### Descarga del dataset y carga a Amazon Simple Storage Service (Amazon S3)

In [2]:
!ls

note_book.ipynb


In [3]:
import boto3
import pandas as pd

# Ruta local de los datos
local_data_path = "../../data/raw"
s3 = boto3.client("s3")

region = sagemaker_session.boto_region_name
f"s3://{bucket}/{input_prefix}"
#input_data = "s3://sagemaker-sample-data-{}/{input_prefix}".format(region)
input_data = f"s3://{bucket}/{input_prefix}".format(region)
print(f"Usando ruta S3: {input_data}")

# Uploading the training data to S3
s3_data_uri = sagemaker_session.upload_data(
    path=local_data_path,
    bucket=bucket,
    key_prefix=input_prefix)
print(f" Datos cargados a: {s3_data_uri}\n")

!aws s3 cp $input_data .

# Verifiacación de datos subidos
response = sagemaker_session.boto_session.client("s3").list_objects_v2(
        Bucket=bucket, 
        Prefix=input_prefix
    )
print(f"\n Archivos en S3:")
if "Contents" in response:
    for obj in response["Contents"]:
        print(f"  ok {obj['Key']}")
        

Usando ruta S3: s3://sagemaker-us-east-1-956463123241/sagemaker/processing-data/input/raw/
 Datos cargados a: s3://sagemaker-us-east-1-956463123241/sagemaker/processing-data/input/raw/

fatal error: An error occurred (404) when calling the HeadObject operation: Key "sagemaker/processing-data/input/raw/" does not exist

 Archivos en S3:
  ok sagemaker/processing-data/input/raw//item_categories.csv
  ok sagemaker/processing-data/input/raw//item_categories_en.csv
  ok sagemaker/processing-data/input/raw//items.csv
  ok sagemaker/processing-data/input/raw//items_en.csv
  ok sagemaker/processing-data/input/raw//sales_train.csv
  ok sagemaker/processing-data/input/raw//shops.csv
  ok sagemaker/processing-data/input/raw//shops_en.csv
  ok sagemaker/processing-data/input/raw//submission.csv
  ok sagemaker/processing-data/input/raw//test.csv


## Construcción del container {#container}

El container BYOC es una imagen Python slim con scikit-learn, pandas y numpy.
No requiere ningún proceso de bootstrapping — SageMaker inyecta y ejecuta el script
directamente con `python3`.

In [4]:
!pwd

/home/sagemaker-user/Arquitectura/processing/notebooks


In [5]:
%cd ../container
!docker build --network sagemaker -t sagemaker-sklearn-preprocess .
%cd ../

/home/sagemaker-user/Arquitectura/processing/container
DEPRECATED: The legacy builder is deprecated and will be removed in a future release.
            BuildKit is currently disabled; enable it by removing the DOCKER_BUILDKIT=0
            environment-variable.

Sending build context to Docker daemon  14.85kB
Step 1/5 : FROM python:3.11-slim
 ---> 7c68b5683872
Step 2/5 : ENV PYTHONHASHSEED 0
 ---> Using cache
 ---> a6dc591cb8da
Step 3/5 : ENV PYTHONIOENCODING UTF-8
 ---> Using cache
 ---> afda23b8c63c
Step 4/5 : RUN pip install --no-cache-dir     numpy==2.4.2     pandas==3.0.0     scikit-learn==1.8.0
 ---> Using cache
 ---> 71e133c26068
Step 5/5 : LABEL com.amazon.studio.user.resources=true
 ---> Using cache
 ---> eef051ce9e5a
Successfully built eef051ce9e5a
Successfully tagged sagemaker-sklearn-preprocess:latest
/home/sagemaker-user/Arquitectura/processing


### Push a Amazon ECR

In [6]:
import boto3

account_id = boto3.client("sts").get_caller_identity().get("Account")
region = boto3.session.Session().region_name

ecr_repository = "sagemaker-sklearn-preprocess"
tag = ":latest"
uri_suffix = "amazonaws.com"

sklearn_repository_uri = "{}.dkr.ecr.{}.{}/{}".format(
    account_id, region, uri_suffix, ecr_repository + tag
)

In [7]:
# Create ECR repository and push docker image
!aws ecr get-login-password --region {region} | docker login --username AWS --password-stdin {account_id}.dkr.ecr.{region}.amazonaws.com
!aws ecr create-repository --repository-name $ecr_repository
!docker tag {ecr_repository + tag} $sklearn_repository_uri
!docker push $sklearn_repository_uri

WARNING! Your password will be stored unencrypted in /home/sagemaker-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credential-stores

Login Succeeded

An error occurred (RepositoryAlreadyExistsException) when calling the CreateRepository operation: The repository with name 'sagemaker-sklearn-preprocess' already exists in the registry with id '956463123241'
The push refers to repository [956463123241.dkr.ecr.us-east-1.amazonaws.com/sagemaker-sklearn-preprocess]

337fce6e: Preparing 
322086e6: Preparing 
c31ad48f: Preparing 
0bcd44fe: Preparing 
latest: digest: sha256:ac0e2a947c2b5444700dfed9acb4422a42d93dee813a1a5a70fd86bb941522d6 size: 1372


In [8]:
## Script de preprocessing {#script}

In [9]:
!pwd

/home/sagemaker-user/Arquitectura/processing


In [10]:
!ls container/preprocess.py

container/preprocess.py


In [11]:
## Ejecutar Processing Job con rutas de SageMaker
from sagemaker.processing import ScriptProcessor
from sagemaker.processing import ProcessingInput, ProcessingOutput

# Crear el ScriptProcessor para ejecutar el script de preprocessing
sklearn_processor = ScriptProcessor(
    base_job_name="sklearn-preprocessor",
    image_uri=sklearn_repository_uri,
    command=["python3"],
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",
    max_runtime_in_seconds=1200,
)

sklearn_processor.run(
    code="preprocess.py",
    arguments=["--input-path", "/opt/ml/processing/input"],
    inputs=[
        ProcessingInput(
            source="s3://{}/{}/census-income.csv".format(bucket, input_prefix),
            destination="/opt/ml/processing/input",
        )
    ],
    outputs=[
        ProcessingOutput(
            output_name="train",
            source="/opt/ml/processing/input/train",
            destination="s3://{}/{}/train".format(bucket, input_preprocessed_prefix),
        ),
        ProcessingOutput(
            output_name="test",
            source="/opt/ml/processing/output/validate",
            destination="s3://{}/{}/test".format(bucket, input_preprocessed_prefix),
        ),
    ],
    logs=True,
)

print("Processing Job completado!")

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:16                                                                                   │
│                                                                                                  │
│   13 │   max_runtime_in_seconds=1200,                                                            │
│   14 )                                                                                           │
│   15                                                                                             │
│ ❱ 16 sklearn_processor.run(                                                                      │
│   17 │   code="preprocess.py",                                                                   │
│   18 │   arguments=["--input-path", "/opt/ml/processing/input"],                                 │
│   19 │   inputs=[                                                                                │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline_context.py:346 in wrapper    │
│                                                                                                  │
│   343 │   │   │                                                                                  │
│   344 │   │   │   return _StepArguments(retrieve_caller_name(self_instance), run_func, *args,    │
│   345 │   │                                                                                      │
│ ❱ 346 │   │   return run_func(*args, **kwargs)                                                   │
│   347 │                                                                                          │
│   348 │   return wrapper                                                                         │
│   349                                                                                            │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/processing.py:670 in run                       │
│                                                                                                  │
│    667 │   │   │   None or pipeline step arguments in case the Processor instance is built with  │
│    668 │   │   │   :class:`~sagemaker.workflow.pipeline_context.PipelineSession`                 │
│    669 │   │   """                                                                               │
│ ❱  670 │   │   normalized_inputs, normalized_outputs = self._normalize_args(                     │
│    671 │   │   │   job_name=job_name,                                                            │
│    672 │   │   │   arguments=arguments,                                                          │
│    673 │   │   │   inputs=inputs,                                                                │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/processing.py:319 in _normalize_args           │
│                                                                                                  │
│    316 │   │                                                                                     │
│    317 │   │   self._current_job_name = self._generate_current_job_name(job_name=job_name)       │
│    318 │   │                                                                                     │
│ ❱  319 │   │   inputs_with_code = self._include_code_in_inputs(inputs, code, kms_key)            │
│    320 │   │   normalized_inputs = self._normalize_inputs(inputs_with_code, kms_key)             │
│    321 │   │   normalized_outputs = self._normalize_outputs(outputs)                             │
│    322 │   │   self.arguments = arguments                  

In [ ]:
## Verificar outputs del Processing Job
import pandas as pd

# Crear cliente S3
s3_client = sagemaker_session.boto_session.client("s3")
output_s3_path = f"s3://{bucket}/{output_prefix}"

print(f"Archivos en {output_s3_path}:")
response = s3_client.list_objects_v2(Bucket=bucket, Prefix=output_prefix)

if "Contents" in response:
    for obj in response["Contents"]:
        print(f"  - {obj['Key']}")
        
# Descargar y verificar el archivo procesado
output_file_key = f"{output_prefix}/datos_entreno.parquet"  # Ajustar según el archivo generado
local_output_path = "processed_data.parquet"

try:
    s3_client.download_file(bucket, output_file_key, local_output_path)
    df = pd.read_parquet(local_output_path)
    print(f"\nPrimeras filas del archivo procesado:")
    print(df.head())
    print(f"\nForma del dataset: {df.shape}")
    print(f"Columnas: {df.columns.tolist()}")
except Exception as e:
    print(f"✗ Error: {e}")